# InvokeAI on Google Colab — persistent setup (Google Drive)

Runs the latest InvokeAI release (web UI) on a Colab GPU, with all state stored on
Google Drive so nothing is lost when the runtime stops and you start it again.

**What persists on Drive** (`INVOKEAI_ROOT`, default `/content/drive/MyDrive/invokeai`):
downloaded models, generated images (`outputs`), the database (boards, gallery, workflows)
and `invokeai.yaml`. The InvokeAI package itself is reinstalled into Colab's fast local
disk each fresh runtime (a few minutes) — only the data lives on Drive.

**Downloading models**: do it inside the running app — **Model Manager** lets you paste a
HuggingFace / CivitAI / direct URL and InvokeAI downloads it straight into the Drive root
itself. For CivitAI, add your API key in the Model Manager settings first.

## GPU
Recommended default: **L4 (24 GB)** — the best balance of availability and capability on
Colab Pro, and enough for SDXL and (with offloading) Flux-class models. Step up to
**A100 (40 GB)** only for the largest models (Flux.1/Flux.2, SD 3.5 Large, Qwen Image) at
full size. **T4 (16 GB, free tier)** handles SD1.5/SDXL. Set it in
`Runtime -> Change runtime type` before running.

## How to use
1. Select an **L4** (recommended) or A100 GPU runtime.
2. Fill the fields in the **Start InvokeAI** cell and run it (authorize Google Drive when
   asked). It prints a public `trycloudflare.com` URL and login, then streams the server
   log right in the cell — open the link once you see `Invoke running on ...`.
3. To stop, kill the runtime in Colab: **Runtime → Disconnect and delete runtime**. To
   resume later, just run the cell again; your models and images are still on Drive.

In [ ]:
#@title  Start InvokeAI  { display-mode: "form" }
#@markdown Runs the latest **InvokeAI** on this Colab GPU with all state saved to
#@markdown **Google Drive**, behind a **password-protected** public link. Fill the fields
#@markdown below and run this cell — it prints a `trycloudflare.com` URL and login, then
#@markdown **streams the server log here**. Open the link once you see `Invoke running on`.
#@markdown To stop, kill the runtime (**Runtime → Disconnect and delete runtime**); your
#@markdown models and images stay on Drive.
#@markdown
#@markdown **Google Drive location** ("My Drive" = personal drive, or a Shared Drive name)
DRIVE_NAME = "My Drive"  #@param {type:"string"}
INVOKEAI_FOLDER = "invokeai"  #@param {type:"string"}
#@markdown **Public link login** (leave the password empty to disable the password)
AUTH_USER = "invoke"  #@param {type:"string"}
AUTH_PASSWORD = "invoke"  #@param {type:"string"}
#@markdown **xformers**: memory-efficient attention (saves VRAM). Off by default =
#@markdown faster launch (reuses Colab's torch); on = installs its own torch, slower.
USE_XFORMERS = False  #@param {type:"boolean"}
#@markdown **Keep-alive**: every 60 s send a light request to the server so Colab is less
#@markdown likely to drop the session for inactivity (best-effort; see Notes).
KEEP_ALIVE = True  #@param {type:"boolean"}

import os
import sys
import re
import time
import subprocess
import urllib.request

# --- Environment check ---
print(f"Python: {sys.version.split()[0]}")
if not (3, 11) <= sys.version_info[:2] <= (3, 12):
    print("WARNING: InvokeAI needs Python 3.11-3.12; this runtime may fail to install.")
try:
    gpu = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip()
    print(f"GPU: {gpu}")
    if not any(name in gpu for name in ("L4", "A100")):
        print("WARNING: for image generation pick 'L4 GPU' (recommended) or 'A100' in "
              "Runtime -> Change runtime type. A T4 only comfortably handles SD1.5/SDXL.")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("ERROR: No NVIDIA GPU. Set Runtime -> Change runtime type, then rerun.")

# --- Mount Google Drive and pin INVOKEAI_ROOT there for persistence ---
from google.colab import drive
drive.mount("/content/drive")

# "My Drive" is your personal drive; any other value is treated as a Shared Drive name.
drive_name = DRIVE_NAME.strip()
if drive_name.lower().replace(" ", "") in ("mydrive", ""):
    drive_base = "/content/drive/MyDrive"
else:
    drive_base = f"/content/drive/Shareddrives/{drive_name}"
if not os.path.isdir(drive_base):
    raise FileNotFoundError(
        f"Drive path not found: {drive_base}. Set DRIVE_NAME to 'My Drive' for your "
        f"personal drive, or to the exact name of a Shared Drive you can access."
    )

os.environ["INVOKEAI_ROOT"] = os.path.join(drive_base, INVOKEAI_FOLDER)
os.environ["INVOKEAI_HOST"] = "127.0.0.1"
os.environ["INVOKEAI_PORT"] = "9090"
os.makedirs(os.environ["INVOKEAI_ROOT"], exist_ok=True)
print(f"INVOKEAI_ROOT = {os.environ['INVOKEAI_ROOT']}")

# --- Install the latest InvokeAI (skip if already present this session) ---
try:
    import invokeai.version
    print(f"InvokeAI already installed: {invokeai.version.__version__}")
except ImportError:
    if USE_XFORMERS:
        # The [xformers] extra pins a matching torch build, so let pip resolve torch itself.
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--upgrade", "invokeai[xformers]"],
            check=True,
        )
    else:
        # Plain install reuses Colab's preinstalled torch (it satisfies torch>=2.7,<3),
        # skipping the multi-GB torch reinstall -> much faster startup.
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--upgrade", "invokeai"],
            check=True,
        )

# --- Fix Colab's protobuf: diffusers needs >=5.26 ('runtime_version'); Colab ships older ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf>=5.26,<6"], check=True)

# --- Download cloudflared and caddy (idempotent) ---
if not os.path.exists("/usr/local/bin/cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "/usr/local/bin/cloudflared",
    )
    os.chmod("/usr/local/bin/cloudflared", 0o755)
if AUTH_PASSWORD and not os.path.exists("/usr/local/bin/caddy"):
    urllib.request.urlretrieve(
        "https://caddyserver.com/api/download?os=linux&arch=amd64", "/usr/local/bin/caddy"
    )
    os.chmod("/usr/local/bin/caddy", 0o755)

port = os.environ["INVOKEAI_PORT"]
proxy_port = "9091"

# --- Optional HTTP Basic Auth in front of InvokeAI via a Caddy reverse proxy ---
if AUTH_PASSWORD:
    pw_hash = subprocess.run(
        ["/usr/local/bin/caddy", "hash-password", "--plaintext", AUTH_PASSWORD],
        capture_output=True, text=True,
    ).stdout.strip()
    caddyfile = (
        f":{proxy_port} {{\n"
        f"\tbasic_auth {{\n"
        f"\t\t{AUTH_USER} {pw_hash}\n"
        f"\t}}\n"
        f"\treverse_proxy 127.0.0.1:{port}\n"
        f"}}\n"
    )
    with open("/content/Caddyfile", "w") as caddy_conf:
        caddy_conf.write(caddyfile)
    subprocess.Popen(
        ["/usr/local/bin/caddy", "run", "--config", "/content/Caddyfile", "--adapter", "caddyfile"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    tunnel_target = proxy_port
else:
    print("WARNING: AUTH_PASSWORD is empty - the public link will be UNPROTECTED.")
    tunnel_target = port

# --- Public tunnel (cloudflared quick tunnel, no signup); started before the server ---
tunnel_log = "/content/cloudflared.log"
subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{tunnel_target}"],
    stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT,
)
public_url = None
for _ in range(30):
    time.sleep(2)
    with open(tunnel_log) as log_file:
        match = re.search(r"https://[-\w]+\.trycloudflare\.com", log_file.read())
    if match:
        public_url = match.group(0)
        break

print("=" * 70)
print(f"Open InvokeAI at: {public_url or 'not ready - see /content/cloudflared.log'}")
if AUTH_PASSWORD:
    print(f"Login: {AUTH_USER} / {AUTH_PASSWORD}")
print("The link works once you see 'Invoke running on http://127.0.0.1:9090' below.")
print("To stop: Runtime -> Disconnect and delete runtime.")
print("=" * 70)

# --- Optional keep-alive: a light periodic request so the runtime keeps seeing activity ---
if KEEP_ALIVE:
    import threading

    def _keep_alive():
        while True:
            time.sleep(60)
            try:
                urllib.request.urlopen(f"http://127.0.0.1:{port}/api/v1/app/version", timeout=10)
                print(f"[keep-alive] {time.strftime('%H:%M:%S')} ping ok", flush=True)
            except Exception:
                pass

    threading.Thread(target=_keep_alive, daemon=True).start()

# --- Launch InvokeAI in the FOREGROUND so its log streams into this cell ---
!invokeai-web

## Notes

- **Persistence**: everything under `INVOKEAI_ROOT` (default `/content/drive/MyDrive/invokeai`)
  is on Google Drive — models, `outputs`, `databases/invokeai.db` and `invokeai.yaml` all
  survive a runtime stop. Next session just run the Start cell again.
- **Always use the same root**: keep `DRIVE_NAME = My Drive` and `INVOKEAI_FOLDER = invokeai`
  so it resolves to `/content/drive/MyDrive/invokeai` every time. Mixing this with other
  notebooks/paths (e.g. `MyDrive/InvokeAI`) splits the database from the files and causes
  missing-model / 404-thumbnail errors.
- **Logs**: InvokeAI runs in the foreground of the Start cell, so its log streams right
  there. The public link works once you see `Invoke running on http://127.0.0.1:9090`.
- **Keep-alive**: `KEEP_ALIVE` (on by default) pings the server every 60 s so the runtime
  stays active, and prints a `[keep-alive]` heartbeat. Best-effort only — Colab's idle
  disconnect is mainly triggered by the browser tab being inactive, which a notebook cannot
  fully prevent. For a stronger keep-alive, run the browser-console script from the
  repo README ("Preventing idle disconnects"), which clicks Connect every 60 s.
- **Stopping**: kill the runtime in Colab — **Runtime → Disconnect and delete runtime** (or
  Interrupt). Drive autosaves; to be safe, stop generating a few seconds before killing it
  so the database finishes writing.
- **Password**: set `AUTH_USER` / `AUTH_PASSWORD`. A Caddy reverse proxy enforces HTTP Basic
  Auth on `:9091` and cloudflared tunnels that port. Empty password = unprotected link.
- **xformers / startup speed**: off by default for a faster launch — the default installs
  plain `invokeai` and reuses Colab's preinstalled torch (no multi-GB torch reinstall).
  Enable `USE_XFORMERS` for memory-efficient attention (saves VRAM); it pins its own
  torch build, so startup is slower.
- **protobuf**: Start upgrades protobuf to >=5.26 because Colab ships an older build that
  crashes diffusers (`cannot import name 'runtime_version'`), which otherwise leaves the
  server down and the tunnel at 502 Bad Gateway.
- **Downloading models**: use the in-app **Model Manager** — paste a HuggingFace / CivitAI
  / direct URL and InvokeAI downloads it into the Drive root itself.
- **Drive space**: large models (Flux, SD 3.5 Large, Qwen ~40 GB) consume a lot of Drive
  quota — keep enough free space.
- **Tunnel URL** changes every launch (quick tunnels are ephemeral). For a stable URL,
  switch cloudflared for an authenticated ngrok / cloudflare named tunnel.
- **GPU**: **L4 (24 GB)** is the recommended default (good availability, handles SDXL and,
  with offloading, Flux-class models). Use A100 (40 GB) for the largest models at full size;
  T4 (16 GB) suits SD1.5/SDXL.